# 05 — Fine-tuning CamemBERT (LoRA) — Phase 5

> **Objectif** : Remplacer la baseline TF-IDF (F1=0.57 avec heuristiques) par un
> **CamemBERT fine-tuné avec LoRA** sur notre corpus de tickets IT francophones.
>
> | Approche | F1-macro (hors-distribution) |
> |---|---|
> | TF-IDF + heuristiques (03_modeling) | 0.57 |
> | **CamemBERT LoRA (ce notebook)** | **cible ≥ 0.80** |
> | Benchmark externe données réelles (04) | 0.83 |
>
> **LoRA** (Low-Rank Adaptation) : seuls ~0.5 % des paramètres sont entraînés
> → adapté GPU 2 GB · convergence rapide sur petit corpus.
>
> Split **identique** à `03_modeling.ipynb` : GroupShuffleSplit par Titre
> (46 Titres train / 12 Titres test — 0 fuite de données).

## 1. Imports & configuration

In [ ]:
import os, sys, json, warnings
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
os.chdir(ROOT)
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib, matplotlib.pyplot as plt
import seaborn as sns
matplotlib.rcParams['figure.dpi'] = 130
plt.style.use('seaborn-v0_8-whitegrid')

import torch
from transformers import (
    CamembertTokenizerFast,
    CamembertForSequenceClassification,
    TrainingArguments, Trainer,
    EarlyStoppingCallback,
    set_seed,
)
from peft import LoraConfig, get_peft_model, TaskType
from datasets import Dataset
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import (
    f1_score, accuracy_score, classification_report, confusion_matrix
)
import yaml, joblib, mlflow

# ── Device ───────────────────────────────────────────────────────────────────
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
if DEVICE == 'cuda':
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    vram_free = (torch.cuda.get_device_properties(0).total_memory
                 - torch.cuda.memory_allocated()) / 1e9
    print(f'GPU : {torch.cuda.get_device_name(0)}')
    print(f'VRAM totale : {vram_gb:.1f} GB | libre : {vram_free:.1f} GB')
else:
    print('CPU (pas de GPU CUDA disponible)')

FIGURES_DIR = ROOT / 'notebooks' / 'figures'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR  = ROOT / 'models'
MODELS_DIR.mkdir(parents=True, exist_ok=True)

with open(ROOT / 'params.yaml') as f:
    params = yaml.safe_load(f)

RS = params['data']['random_state']
set_seed(RS)
print(f'\nRacine projet : {ROOT}')
print(f'Device        : {DEVICE}')
print(f'PyTorch       : {torch.__version__}')

## 2. Chargement & Split anti-fuite (identique à 03_modeling)

Même **GroupShuffleSplit par Titre** que dans `03_modeling.ipynb` :
46 Titres entraînement / 12 Titres test → aucun Titre de test vu en entraînement.

In [ ]:
df = pd.read_csv(ROOT / params['data']['processed_path'])
print(f'Dataset : {len(df)} tickets | {df["Catégorie"].nunique()} classes')
print(f'Titres uniques : {df["Titre"].nunique()}')

# Encodage labels
classes    = sorted(df['Catégorie'].unique())
label2id   = {c: i for i, c in enumerate(classes)}
id2label   = {i: c for c, i in label2id.items()}
df['label'] = df['Catégorie'].map(label2id)
NUM_LABELS  = len(classes)
print(f'Classes ({NUM_LABELS}) : {classes}')

# ── GroupShuffleSplit par Titre ───────────────────────────────────────────────
titres_uniq = df.groupby('Titre')['Catégorie'].first().reset_index()
sss = StratifiedShuffleSplit(n_splits=1,
                             test_size=params['data']['test_size'],
                             random_state=RS)
for tr_idx, te_idx in sss.split(titres_uniq['Titre'], titres_uniq['Catégorie']):
    train_titres = set(titres_uniq.iloc[tr_idx]['Titre'])
    test_titres  = set(titres_uniq.iloc[te_idx]['Titre'])

train_df = df[df['Titre'].isin(train_titres)].copy().reset_index(drop=True)
test_df  = df[df['Titre'].isin(test_titres)].copy().reset_index(drop=True)

print(f'\nOverlap Titres : {len(train_titres & test_titres)} (doit etre 0)')
print(f'Train : {len(train_df)} tickets | {len(train_titres)} Titres')
print(f'Test  : {len(test_df)} tickets | {len(test_titres)} Titres')
print(f'\nTitres en test :')
for t in sorted(test_titres):
    cat = titres_uniq[titres_uniq['Titre'] == t]['Catégorie'].values[0]
    print(f'  [{cat}] {t}')

## 3. Tokenisation

In [ ]:
MODEL_NAME = 'camembert-base'
MAX_LEN    = 128   # tickets courts — suffisant

tokenizer = CamembertTokenizerFast.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(
        batch['text_clean'],
        padding='max_length',
        truncation=True,
        max_length=MAX_LEN,
    )

# Convertir en HuggingFace Dataset
cols_keep = ['text_enriched', 'label']
# Renommer text_enriched -> text_clean pour la tokenisation
train_df2 = train_df[cols_keep].rename(columns={'text_enriched':'text_clean'})
test_df2  = test_df[cols_keep].rename(columns={'text_enriched':'text_clean'})
ds_train = Dataset.from_pandas(train_df2)
ds_test  = Dataset.from_pandas(test_df2)

ds_train = ds_train.map(tokenize, batched=True, batch_size=64)
ds_test  = ds_test.map(tokenize,  batched=True, batch_size=64)

ds_train = ds_train.rename_column('label', 'labels')
ds_test  = ds_test.rename_column('label', 'labels')
ds_train.set_format('torch', columns=['input_ids','attention_mask','labels'])
ds_test.set_format('torch',  columns=['input_ids','attention_mask','labels'])

# Split train → train + validation (10 %)
ds_split = ds_train.train_test_split(test_size=0.1, seed=RS)
ds_train_final = ds_split['train']
ds_val         = ds_split['test']

print(f'Train final : {len(ds_train_final)} | Val : {len(ds_val)} | Test : {len(ds_test)}')
print(f'Tokenizer   : {MODEL_NAME} | max_length = {MAX_LEN}')

## 4. Modèle CamemBERT + LoRA

**LoRA** (Low-Rank Adaptation) :
- Paramètres totaux CamemBERT : ~110 M (gelés)
- Paramètres LoRA entraînés : ~0.3 M seulement
- Adapté GPU 2 GB · pas de dégradation catastrophique

In [ ]:
# Charger CamemBERT avec tête de classification
dtype = torch.float32  # LoRA + fp16 scaler incompatibles -> toujours FP32

model = CamembertForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels    = NUM_LABELS,
    id2label      = id2label,
    label2id      = label2id,
    torch_dtype   = dtype,
    ignore_mismatched_sizes = True,
)

# ── LoRA config ───────────────────────────────────────────────────────────────
lora_cfg = LoraConfig(
    task_type        = TaskType.SEQ_CLS,
    r                = 8,          # rang (plus grand = plus expressif)
    lora_alpha       = 16,         # scale = lora_alpha / r = 2
    lora_dropout     = 0.1,
    target_modules   = ['query', 'value'],  # couches attention CamemBERT
    bias             = 'none',
    inference_mode   = False,
)

model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()

# Déplacer sur device
model = model.to(DEVICE)

# Stats mémoire GPU
if DEVICE == 'cuda':
    used = torch.cuda.memory_allocated() / 1e6
    total_vram = torch.cuda.get_device_properties(0).total_memory / 1e6
    print(f'\nGPU mémoire utilisée : {used:.0f} MB / {total_vram:.0f} MB')

## 5. Entraînement

In [ ]:
# Métrique d'évaluation
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    f1  = f1_score(labels, preds, average='macro', zero_division=0)
    acc = accuracy_score(labels, preds)
    return {'f1_macro': f1, 'accuracy': acc}

# ── Hyperparamètres (adaptés 2 GB VRAM) ──────────────────────────────────────
BS_TRAIN = 2 if DEVICE == 'cuda' else 8  # FP32 double la memoire -> batch 2
GRAD_ACC = 4                       # effective batch = 16
EPOCHS   = 10
LR       = 2e-4

OUTPUT_DIR = str(ROOT / 'models' / 'camembert_lora')

args = TrainingArguments(
    output_dir                  = OUTPUT_DIR,
    num_train_epochs            = EPOCHS,
    per_device_train_batch_size = BS_TRAIN,
    per_device_eval_batch_size  = 16,
    gradient_accumulation_steps = GRAD_ACC,
    learning_rate               = LR,
    warmup_ratio                = 0.1,
    weight_decay                = 0.01,
    lr_scheduler_type           = 'cosine',
    fp16                        = False,  # LoRA params FP32 incompatibles avec scaler FP16
    gradient_checkpointing      = (DEVICE == 'cuda'),
    eval_strategy               = 'epoch',
    save_strategy               = 'epoch',
    load_best_model_at_end      = True,
    metric_for_best_model       = 'f1_macro',
    greater_is_better           = True,
    logging_steps               = 10,
    report_to                   = 'none',
    seed                        = RS,
    dataloader_pin_memory       = False,   # evite OOM sur petit GPU
)

trainer = Trainer(
    model            = model,
    args             = args,
    train_dataset    = ds_train_final,
    eval_dataset     = ds_val,
    compute_metrics  = compute_metrics,
    callbacks        = [EarlyStoppingCallback(early_stopping_patience=3)],
)

print(f'Lancement entrainement : {EPOCHS} epochs | LR={LR} | batch_eff={BS_TRAIN*GRAD_ACC}')
print(f'Device : {DEVICE} | FP16 : {DEVICE == "cuda"} | grad_checkpoint : {DEVICE == "cuda"}')
print()
train_result = trainer.train()
print(f'\nEntrainement termine. Steps : {train_result.global_step}')

## 6. Évaluation sur le jeu de test

In [ ]:
# Prédictions sur le test set
preds_out = trainer.predict(ds_test)
logits    = preds_out.predictions
y_true    = preds_out.label_ids
y_pred    = np.argmax(logits, axis=-1)

# Scores
f1_camembert  = f1_score(y_true, y_pred, average='macro', zero_division=0)
acc_camembert = accuracy_score(y_true, y_pred)

print(f'CamemBERT LoRA — F1-macro : {f1_camembert:.4f} | Accuracy : {acc_camembert:.4f}')
print()
print(classification_report(y_true, y_pred,
                             target_names=classes, zero_division=0))

In [ ]:
# Matrice de confusion
cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(pd.DataFrame(cm, index=classes, columns=classes),
            annot=True, fmt='d', cmap='Blues',
            linewidths=.5, linecolor='white', ax=ax)
ax.set_title(f'CamemBERT LoRA — Matrice de confusion (F1={f1_camembert:.3f})',
             fontweight='bold')
ax.set_ylabel('Réel'); ax.set_xlabel('Prédit')
plt.tight_layout()
fig.savefig(FIGURES_DIR / 'cam_01_confusion.png', dpi=150, bbox_inches='tight')
plt.show()
print('cam_01_confusion.png')

## 7. Analyse de confiance (softmax → probabilités)

In [ ]:
import torch.nn.functional as F

# Probabilités softmax depuis les logits
probs    = torch.softmax(torch.tensor(logits, dtype=torch.float32), dim=-1).numpy()
conf_max = probs.max(axis=1)
correct  = (y_pred == y_true)

THRESHOLD = 0.40
haute = conf_max >= THRESHOLD
basse = conf_max <  THRESHOLD

print(f'Avec seuil = {THRESHOLD:.0%} :')
print(f'  Auto-route  : {haute.sum()} tickets ({haute.mean()*100:.1f}%) '
      f'-> precision {correct[haute].mean():.2%}')
print(f'  Revue humaine (HITL) : {basse.sum()} tickets ({basse.mean()*100:.1f}%)')

# Figure distribution confiance
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(conf_max, bins=25, color='#4C72B0', edgecolor='white', alpha=0.85)
ax.axvline(THRESHOLD, color='orange', linestyle='--', linewidth=2,
           label=f'Seuil HITL = {THRESHOLD}')
ax.set_xlabel('Score de confiance (softmax max)')
ax.set_title('CamemBERT LoRA — Distribution de la confiance', fontweight='bold')
ax.legend()
plt.tight_layout()
fig.savefig(FIGURES_DIR / 'cam_02_confiance.png', dpi=150, bbox_inches='tight')
plt.show()
print('cam_02_confiance.png')

## 8. Tableau comparatif — Baseline vs CamemBERT

In [ ]:
synthese = pd.DataFrame({
    'Modèle': [
        'Dummy (référence plancher)',
        'TF-IDF + LR+SMOTE',
        'TF-IDF + LinearSVC',
        'TF-IDF + LightGBM',
        'TF-IDF + LinearSVC + Heuristiques',
        'CamemBERT LoRA (ce notebook)',
    ],
    'F1-macro': [0.0519, 0.3746, 0.3753, 0.3746, 0.5673, round(f1_camembert, 4)],
    'Approche': [
        'Règle', 'ML', 'ML', 'ML', 'ML + règles métier', 'Transformeur FR (LoRA)'
    ],
})
synthese = synthese.sort_values('F1-macro').reset_index(drop=True)

print('=' * 68)
print('COMPARAISON — TF-IDF baseline vs CamemBERT LoRA')
print('=' * 68)
print(synthese.to_string(index=False))
print('=' * 68)
gain = f1_camembert - 0.5673
print(f'\nGain CamemBERT vs meilleur TF-IDF+heuristiques : +{gain:.4f}')

# Figure benchmark
fig, ax = plt.subplots(figsize=(11, 5))
colors = ['#C44E52' if 'CamemBERT' in n else '#4C72B0'
          for n in synthese['Modèle']]
bars = ax.barh(synthese['Modèle'], synthese['F1-macro'],
               color=colors, edgecolor='white', height=0.6)
ax.axvline(0.80, color='green', linestyle='--', linewidth=1.5, label='Cible ≥ 0.80')
ax.axvline(0.5673, color='orange', linestyle=':', linewidth=1.5,
           label='Meilleur TF-IDF+heuristiques (0.57)')
for bar, val in zip(bars, synthese['F1-macro']):
    ax.text(bar.get_width() + .008, bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', fontsize=9)
ax.set_xlim(0, 1.05)
ax.set_xlabel('F1-macro (split anti-fuite GroupShuffleSplit par Titre)')
ax.set_title('Benchmark — TF-IDF baseline vs CamemBERT LoRA\n'
             '(rouge = CamemBERT, bleu = TF-IDF)', fontweight='bold')
ax.legend()
plt.tight_layout()
fig.savefig(FIGURES_DIR / 'cam_03_benchmark.png', dpi=150, bbox_inches='tight')
plt.show()
print('cam_03_benchmark.png')

## 9. Sauvegarde de l'adaptateur LoRA + métadonnées

In [ ]:
# Sauvegarder l'adaptateur LoRA (pas le modèle entier)
adapter_path = ROOT / 'models' / 'camembert_lora_adapter'
model.save_pretrained(str(adapter_path))
tokenizer.save_pretrained(str(adapter_path))
print(f'Adaptateur LoRA sauvegarde : {adapter_path}')
print(f'Fichiers : {[f.name for f in adapter_path.iterdir()]}')

# Métadonnées
meta_cam = {
    'model_name'            : 'CamemBERT-base + LoRA',
    'base_model'            : 'camembert-base',
    'lora_r'                : 8,
    'lora_alpha'            : 16,
    'lora_target_modules'   : ['query', 'value'],
    'max_length'            : MAX_LEN,
    'epochs_trained'        : int(train_result.global_step / (len(ds_train_final) // (BS_TRAIN * GRAD_ACC) + 1)),
    'f1_macro'              : round(float(f1_camembert), 4),
    'accuracy'              : round(float(acc_camembert), 4),
    'f1_baseline_tfidf'     : 0.3753,
    'f1_baseline_heuristiques': 0.5673,
    'gain_vs_baseline'      : round(float(f1_camembert) - 0.5673, 4),
    'split_strategy'        : 'GroupShuffleSplit par Titre (46 train / 12 test)',
    'confidence_threshold'  : THRESHOLD,
    'hitl_auto_pct'         : round(float(haute.mean()) * 100, 1),
    'hitl_auto_precision'   : round(float(correct[haute].mean()), 4) if haute.sum() > 0 else None,
    'classes'               : classes,
    'train_size'            : len(ds_train_final),
    'val_size'              : len(ds_val),
    'test_size'             : len(ds_test),
    'device'                : DEVICE,
    'torch_dtype'           : str(dtype).replace('torch.', ''),
    'feature'               : 'text_enriched (type + priorite simplifiees + titre + description)',
    'f1_macro_sans_enrichissement' : 0.5325,
}
meta_path = ROOT / 'models' / 'camembert_metadata.json'
with open(meta_path, 'w', encoding='utf-8') as f:
    json.dump(meta_cam, f, ensure_ascii=False, indent=2)
print(f'Metadonnees : {meta_path}')
print(json.dumps(meta_cam, ensure_ascii=False, indent=2))

## 10. Log MLflow

In [ ]:
(ROOT / 'mlruns').mkdir(parents=True, exist_ok=True)
mlflow.set_tracking_uri((ROOT / 'mlruns').as_uri())
mlflow.set_experiment('helpdesk-classification')

with mlflow.start_run(run_name='CamemBERT_LoRA'):
    mlflow.log_params({
        'model'           : 'camembert-base',
        'lora_r'          : 8,
        'lora_alpha'      : 16,
        'target_modules'  : 'query,value',
        'max_length'      : MAX_LEN,
        'learning_rate'   : LR,
        'epochs'          : EPOCHS,
        'batch_effective' : BS_TRAIN * GRAD_ACC,
        'device'          : DEVICE,
    })
    mlflow.log_metrics({
        'f1_macro'    : f1_camembert,
        'accuracy'    : acc_camembert,
        'gain_vs_baseline': f1_camembert - 0.5673,
    })
    mlflow.log_artifact(str(meta_path))
    print('MLflow run enregistre.')

## 11. Résumé final

In [ ]:
print('=' * 65)
print('RESUME — PHASE 5 : CamemBERT LoRA')
print('=' * 65)
print(f'  Modele base         : camembert-base')
print(f'  LoRA r/alpha        : 8 / 16 | modules : query, value')
print(f'  F1-macro            : {f1_camembert:.4f}')
print(f'  Accuracy            : {acc_camembert:.4f}')
print(f'  Gain vs TF-IDF+heuri: +{f1_camembert - 0.5673:.4f}')
print(f'  HITL auto-route     : {haute.mean()*100:.1f}% @ {correct[haute].mean():.0%} precision')
print()
print('  Figures : notebooks/figures/cam_01_confusion.png')
print('            notebooks/figures/cam_02_confiance.png')
print('            notebooks/figures/cam_03_benchmark.png')
print()
print('  Adaptateur LoRA : models/camembert_lora_adapter/')
print('  Metadonnees     : models/camembert_metadata.json')
print()
cible = 0.80
atteint = f1_camembert >= cible
print(f'  Objectif PFE (F1 >= {cible}) : {"ATTEINT" if atteint else "Non atteint — voir Phase 5 optimisations"}')
print()
print('  -> Etape suivante : Phase 6 — RAG semantique (embeddings FR + vector DB)')